---
How to Improve Accuracy
---

From your notebook, you're already using a good baseline:

* ✅ Train/Validation/Test split
* ✅ Multi-layer ANN (`128 → 64 → 32 → 1`)
* ✅ ReLU hidden layers
* ✅ Sigmoid output
* ✅ Adam optimizer
* ✅ Early Stopping
* ✅ Binary Cross Entropy

However, there are several reasons your model may not achieve high accuracy.

---

# 1. Fix the train-test split (Important)

I noticed this in your notebook:

```python
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=20,
    random_state=42
)
```

Using

```python
test_size=20
```

means **20 samples**, **not 20%**.

Instead use

```python
test_size=0.2
```

or

```python
test_size=0.25
```

This gives a better representation of unseen data.

---

# 2. Normalize the features

ANNs perform much better when every feature has a similar scale.

Example

```
Age
20-60

Salary
10,000-1,000,000

Balance
0-5000
```

The network focuses more on larger numbers.

Use

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_valid = scaler.transform(X_valid)

X_test = scaler.transform(X_test)
```

This alone often improves ANN performance.

---

# 3. Use a better architecture

Current model

```python
128
64
32
1
```

Try

```python
model = Sequential([
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])
```

Sometimes a **smaller network generalizes better** than a large one.

---

# 4. Add Batch Normalization

Why?

During training,

```
Layer 1 outputs

↓

Layer 2 inputs change

↓

Layer 3 changes

↓

Training becomes unstable
```

Batch Normalization keeps activations centered.

Example

```python
from tensorflow.keras.layers import BatchNormalization

model = Sequential([
    Dense(64, activation='relu'),
    BatchNormalization(),

    Dense(32, activation='relu'),
    BatchNormalization(),

    Dense(16, activation='relu'),
    BatchNormalization(),

    Dense(1, activation='sigmoid')
])
```

Benefits

* faster training
* more stable gradients
* higher accuracy

---

# 5. Add Dropout

If training accuracy is much higher than validation accuracy,

```
Training Accuracy = 99%

Validation Accuracy = 84%
```

the model is memorizing the training data.

Dropout randomly disables neurons during training.

Example

```python
from tensorflow.keras.layers import Dropout

model = Sequential([
    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(32, activation='relu'),
    Dropout(0.3),

    Dense(16, activation='relu'),

    Dense(1, activation='sigmoid')
])
```

Typical values

```
0.2

0.3

0.5
```

---

# 6. Reduce learning rate

Your notebook uses

```python
Adam(learning_rate=0.01)
```

This is relatively high.

Try

```python
Adam(learning_rate=0.001)
```

or

```python
Adam(learning_rate=0.0005)
```

Large learning rates can cause the optimizer to overshoot the best solution.

---

# 7. Increase epochs

You currently train for

```python
epochs=10
```

This is often too few.

Try

```python
epochs=100
```

with

```python
EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)
```

Training stops automatically when validation performance stops improving.

---

# 8. Handle class imbalance

Check

```python
y.value_counts()
```

Example

```
No : 900

Yes:100
```

The model can predict "No" for every sample and still get **90% accuracy**.

Solutions

```python
class_weight
```

or

```python
SMOTE
```

or

```python
RandomOverSampler
```

---

# 9. Tune the decision threshold

Most people use

```python
0.5
```

But suppose predictions are

```
0.42

0.44

0.46

0.49
```

Using 0.5 classifies all as negative.

Try

```python
0.4
```

or

```python
0.35
```

Choose the threshold that gives the best F1 score or business objective.

---

# 10. Evaluate more than accuracy

Instead of

```python
accuracy
```

also compute

```python
Precision

Recall

F1-score

ROC-AUC
```

For binary classification:

```python
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score

pred = model.predict(X_test)

pred = (pred > 0.5).astype(int)

print(classification_report(y_test, pred))

print(confusion_matrix(y_test, pred))

print(roc_auc_score(y_test, pred))
```

This helps identify whether errors are due to false positives or false negatives rather than relying only on accuracy.

---

# 11. Hyperparameter tuning

Experiment with:

* Number of layers: 2–5
* Neurons: 16, 32, 64, 128
* Learning rate: 0.1, 0.01, 0.001, 0.0001
* Batch size: 16, 32, 64
* Activation: ReLU, LeakyReLU, ELU
* Optimizer: Adam, RMSprop, SGD
* Dropout: 0.2, 0.3, 0.5

Tools such as **KerasTuner** can automate this search.

---

# 12. Feature engineering

Model performance is limited by the information in the features.

Examples:

* Encode categorical variables correctly
* Remove irrelevant features
* Create interaction features
* Handle missing values
* Remove outliers where appropriate
* Normalize numerical features

Improving features often yields larger gains than changing the neural network.

---

## Recommended improved model

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(16, activation='relu'),

    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_valid, y_valid),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop]
)
```

### Recommended improvement order

1. Fix `test_size` to `0.2`.
2. Apply `StandardScaler`.
3. Reduce the learning rate to `0.001`.
4. Train for up to 100 epochs with `EarlyStopping`.
5. Add `BatchNormalization`.
6. Add `Dropout` if you observe overfitting.
7. Check for class imbalance and use class weights or resampling if needed.
8. Evaluate with Precision, Recall, F1-score, ROC-AUC, and a confusion matrix.
9. Tune hyperparameters.
10. Improve feature engineering, as it usually has the biggest impact on final accuracy.
